In [2]:
import pandas as pd
import torch
import numpy as np
import re
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    pipeline
)
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

C:\Users\ACOYLO1\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
df = pd.read_excel(r'Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\EXPORT PRESTACIONES.xlsx')
df['oficina'] = df['ES - Unit ID Text'].str.replace('mapfre_es_', '', regex=False)
df = df[df['ES - Touchpoint'] == 'Prestación Salud - Asistencia']
df['ES - Health - Medical specialty'] = pd.to_numeric(df['ES - Health - Medical specialty'], errors='coerce')
df['ES - Health - Concerted Service Code'] = pd.to_numeric(df['ES - Health - Concerted Service Code'], errors='coerce')
df.head()

,ES - Client first name,ES - Client last name,Email,ES - Client ID,ES - Phone number,ES - DNI,ES - Province,ES - BirthDate,ES - General segment,Survey Status,ES - Touchpoint,ES - MAPFRE Santander Record,ES - Policy number,ES - Product type,DR,Export AT Description,ES - Unit ID Text,ES - Unit ID,ES - Agent,Surveyid for internal use (e.g. RI link),Fecha de respuesta,ES - NPS including Corredores,ES - Corredores - Verbatim Comment including Corredores,ES - Contact Intrest including Corredores,ES - General Claim - Type of claim Code,ES - General Claim - Type of claim Descripction,ES - General Claim - Type of claim,ES - General Claim - Processor Code,ES - General Claim - Processor Name,ES - General Claim - Receiving center,ES - General Claim - Center Code,ES - General Claim - Center user,ES - General Claim - File Number,ES - General Claim - File open date,ES - General Claim - File closure date,ES - Auto - Number Plate,ES - Auto - Sinister Province,ES - Auto - Distinguished Garage,ES - Auto - Supplier Province - String,ES - Auto - Type management - String,ES - Auto - Estimated Time - String,ES - Auto - Cause,ES - Auto - Cause description,ES - Auto - Supplier Code,ES - Auto - Taxi Supplier Code,ES - Auto - Towing Supplier Code,ES - Auto - Repair Shop Supplier Code,ES - Auto - Supplier Name,ES - Auto - Taxi Supplier Name,ES - Auto - Towing Supplier Name,ES - Auto - Repair Shop Supplier Name,ES - Auto - Towing,ES - Auto - Repair Shop,ES - Auto - Taxi,ES - Auto - Supplier Type Text,ES - Motivo de valoración - Auto Assitance - Corredores,ES - Auto Assitance - Request - Corredores,ES - Detailed Reason - Auto Assitance - Service -Corredores,ES - Detailed Reason - Auto Assitance - Towingtaxi -Corredores,ES - Auto Assitance - Satisfaction - Ease,ES - Auto Assitance - Satisfaction - Info - Corr,ES - Auto Assitance - Satisfaction - Towing - Time - Corr,ES - Auto Assitance - Satisfaction - Taxi - Time - Corr,ES - Auto Assitance - Satisfaction - Towing - Driver - Corr,ES - Auto Assitance - Satisfaction - Kidness - Taxi - Corr,ES - Auto Assitance - Satisfaction - Time -Corr,ES - Motivo de valoración - Auto Claim - Corredores,ES - Claim - Detailed reason - Communication New,ES - Auto -Claim - Detailed reason - Reparacion,ES - Auto -Claim - Detailed reason - Information,ES - Auto -Claim - Detailed reason - Resolution,ES - Auto Claim - Satisfaction Atention,ES - Auto Claim - Satisfaction Next Steps,ES - Auto Claim - Satisfaction Proactivity,ES - Auto Claim - Satisfaction Explanations,ES - Auto Claim - Satisfaction Assesment,ES - Auto Claim - Satisfaction Resolution,ES - Auto Claim - Satisfaction Quality,ES - Auto Claim - Satisfaction Repairer,ES - Home - Compensation Amount,ES - Home - File paid Amount,ES - Home - Number repair Services,ES - Home - Activity Repair Code,ES - Home - Number Control Services,ES - Home - Code Control Activity,ES - Home - Group Tramitation Code,ES - Home - Risk Address Postal Code,ES - Home - Risk Address Town,ES - Home - Risk Address Province,ES - Commerce - Epigraph code,ES - Commerce - Epigraph name,ES - Commerce - Mapfre entity,ES - Commerce - Mapfre Entity Code,ES - Commerce - Santander purchase channel,ES - Commerce - Santander user ID,ES - Commerce - Dossier Zone,ES - Commerce - Expert Name,ES - Commerce - Expert Code,ES - Commerce - Repairer Name,ES - Commerce - Repairer Code,ES - Commerce - PZCIS Code,ES- Motivo de valoración - Hogar - Corredores,ES - Home - Detailed Reason Claim Communication including Corredores,ES - Home - Detailed Reason Claim Evolution including Corredores,ES - Home - Satisfaction Repairers Service including Corredores,ES - Detailed Reason Claim Resolution including Corr,ES - Detailed Reason Compensation Charge including Corredores,ES - Home - Satisfaction Claim Management including Corredores,ES - Home - Satisfaction Information including Corredores,ES - Home - Satisfaction Proactivity including Corredores,ES - Home - Satisfaction Communication including Corredores,ES - Home - Satisfaction 

In [4]:
encuestas = df['Surveyid for internal use (e.g. RI link)'].nunique()
print(f"Total Encuestas: {encuestas}")

Total Encuestas: 14477


In [5]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
df_especialidad = pd.read_csv(r'Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\NPS_Salud\Tablas\Especialidad Realizadora del Acto Salud.txt', encoding="UTF-16", sep="\t")
df_especialidad.head()

,Código Especialidad Realizadora Acto,Nombre Especialidad Realizadora Acto
0,0,NO APLICA
1,1,MEDICINA GENERAL
2,2,PEDIATRIA Y PUERICULTURA
3,3,ALERGOLOGIA
4,4,ANALISIS CLINICOS


In [6]:
# Realizar el cruce entre export_nps y df_estructura 
df = df.merge(df_especialidad, left_on='ES - Health - Medical specialty', right_on='Código Especialidad Realizadora Acto', how='inner')
# Mostrar el resultado
df.head()

,ES - Client first name,ES - Client last name,Email,ES - Client ID,ES - Phone number,ES - DNI,ES - Province,ES - BirthDate,ES - General segment,Survey Status,ES - Touchpoint,ES - MAPFRE Santander Record,ES - Policy number,ES - Product type,DR,Export AT Description,ES - Unit ID Text,ES - Unit ID,ES - Agent,Surveyid for internal use (e.g. RI link),Fecha de respuesta,ES - NPS including Corredores,ES - Corredores - Verbatim Comment including Corredores,ES - Contact Intrest including Corredores,ES - General Claim - Type of claim Code,ES - General Claim - Type of claim Descripction,ES - General Claim - Type of claim,ES - General Claim - Processor Code,ES - General Claim - Processor Name,ES - General Claim - Receiving center,ES - General Claim - Center Code,ES - General Claim - Center user,ES - General Claim - File Number,ES - General Claim - File open date,ES - General Claim - File closure date,ES - Auto - Number Plate,ES - Auto - Sinister Province,ES - Auto - Distinguished Garage,ES - Auto - Supplier Province - String,ES - Auto - Type management - String,ES - Auto - Estimated Time - String,ES - Auto - Cause,ES - Auto - Cause description,ES - Auto - Supplier Code,ES - Auto - Taxi Supplier Code,ES - Auto - Towing Supplier Code,ES - Auto - Repair Shop Supplier Code,ES - Auto - Supplier Name,ES - Auto - Taxi Supplier Name,ES - Auto - Towing Supplier Name,ES - Auto - Repair Shop Supplier Name,ES - Auto - Towing,ES - Auto - Repair Shop,ES - Auto - Taxi,ES - Auto - Supplier Type Text,ES - Motivo de valoración - Auto Assitance - Corredores,ES - Auto Assitance - Request - Corredores,ES - Detailed Reason - Auto Assitance - Service -Corredores,ES - Detailed Reason - Auto Assitance - Towingtaxi -Corredores,ES - Auto Assitance - Satisfaction - Ease,ES - Auto Assitance - Satisfaction - Info - Corr,ES - Auto Assitance - Satisfaction - Towing - Time - Corr,ES - Auto Assitance - Satisfaction - Taxi - Time - Corr,ES - Auto Assitance - Satisfaction - Towing - Driver - Corr,ES - Auto Assitance - Satisfaction - Kidness - Taxi - Corr,ES - Auto Assitance - Satisfaction - Time -Corr,ES - Motivo de valoración - Auto Claim - Corredores,ES - Claim - Detailed reason - Communication New,ES - Auto -Claim - Detailed reason - Reparacion,ES - Auto -Claim - Detailed reason - Information,ES - Auto -Claim - Detailed reason - Resolution,ES - Auto Claim - Satisfaction Atention,ES - Auto Claim - Satisfaction Next Steps,ES - Auto Claim - Satisfaction Proactivity,ES - Auto Claim - Satisfaction Explanations,ES - Auto Claim - Satisfaction Assesment,ES - Auto Claim - Satisfaction Resolution,ES - Auto Claim - Satisfaction Quality,ES - Auto Claim - Satisfaction Repairer,ES - Home - Compensation Amount,ES - Home - File paid Amount,ES - Home - Number repair Services,ES - Home - Activity Repair Code,ES - Home - Number Control Services,ES - Home - Code Control Activity,ES - Home - Group Tramitation Code,ES - Home - Risk Address Postal Code,ES - Home - Risk Address Town,ES - Home - Risk Address Province,ES - Commerce - Epigraph code,ES - Commerce - Epigraph name,ES - Commerce - Mapfre entity,ES - Commerce - Mapfre Entity Code,ES - Commerce - Santander purchase channel,ES - Commerce - Santander user ID,ES - Commerce - Dossier Zone,ES - Commerce - Expert Name,ES - Commerce - Expert Code,ES - Commerce - Repairer Name,ES - Commerce - Repairer Code,ES - Commerce - PZCIS Code,ES- Motivo de valoración - Hogar - Corredores,ES - Home - Detailed Reason Claim Communication including Corredores,ES - Home - Detailed Reason Claim Evolution including Corredores,ES - Home - Satisfaction Repairers Service including Corredores,ES - Detailed Reason Claim Resolution including Corr,ES - Detailed Reason Compensation Charge including Corredores,ES - Home - Satisfaction Claim Management including Corredores,ES - Home - Satisfaction Information including Corredores,ES - Home - Satisfaction Proactivity including Corredores,ES - Home - Satisfaction Communication including Corredores,ES - Home - Satisfaction 

In [7]:
encuestas = df['Surveyid for internal use (e.g. RI link)'].nunique()
print(f"Total Encuestas: {encuestas}")

Total Encuestas: 14477


In [8]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
df_servicio = pd.read_csv(r'Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\NPS_Salud\Tablas\Servicio Concertado Realizador Acto Salud.txt', encoding="UTF-16", sep="\t")
df_servicio['Código SSCC Realizador Acto'] = pd.to_numeric(df_servicio['Código SSCC Realizador Acto'], errors='coerce')
df_servicio.head()

,Código Provincia Realizador Acto,Nombre Provincia Realizador Acto,Código Grupo Hospitalario Realizador Acto,Nombre Grupo Hospitalario Realizador Acto,Código SSCC Realizador Acto,Nombre SSCC Realizador Acto,Nombre Centro Hospitalario Realizador Acto
0,1.0,ARABA-ALAVA,88.0,RESTO_TITULARES,1002864.0,"INFANTE RIAÑO, RICARDO",NaN
1,1.0,ARABA-ALAVA,88.0,RESTO_TITULARES,1006592.0,"GAJATE MENDIZABAL, SAIOA",NaN
2,1.0,ARABA-ALAVA,4.0,QUIRONSALUD,1006865.0,HOSPITAL QUIRONSALUD VITORIA,1002050551101 - HOSPITAL QUIRONSALUD VITORIA
3,1.0,ARABA-ALAVA,88.0,RESTO_TITULARES,1010115.0,RADIOLOGIA DENTAL OPERA,NaN
4,1.0,ARABA-ALAVA,88.0,RESTO_TITULARES,1013028.0,CLINICA OFTALMOLOGICA MIRANZA OKULAR,NaN


In [9]:
# Realizar el cruce entre export_nps y df_estructura 
# Si devuelve un fallo en el cruce ir al fichero Servicio Concertado Realizador Acto Salud.txt y eliminar los registros que no sean numéricos 
df = df.merge(df_servicio, left_on='ES - Health - Concerted Service Code', right_on='Código SSCC Realizador Acto', how='inner')
# Mostrar el resultado
df.head()

,ES - Client first name,ES - Client last name,Email,ES - Client ID,ES - Phone number,ES - DNI,ES - Province,ES - BirthDate,ES - General segment,Survey Status,ES - Touchpoint,ES - MAPFRE Santander Record,ES - Policy number,ES - Product type,DR,Export AT Description,ES - Unit ID Text,ES - Unit ID,ES - Agent,Surveyid for internal use (e.g. RI link),Fecha de respuesta,ES - NPS including Corredores,ES - Corredores - Verbatim Comment including Corredores,ES - Contact Intrest including Corredores,ES - General Claim - Type of claim Code,ES - General Claim - Type of claim Descripction,ES - General Claim - Type of claim,ES - General Claim - Processor Code,ES - General Claim - Processor Name,ES - General Claim - Receiving center,ES - General Claim - Center Code,ES - General Claim - Center user,ES - General Claim - File Number,ES - General Claim - File open date,ES - General Claim - File closure date,ES - Auto - Number Plate,ES - Auto - Sinister Province,ES - Auto - Distinguished Garage,ES - Auto - Supplier Province - String,ES - Auto - Type management - String,ES - Auto - Estimated Time - String,ES - Auto - Cause,ES - Auto - Cause description,ES - Auto - Supplier Code,ES - Auto - Taxi Supplier Code,ES - Auto - Towing Supplier Code,ES - Auto - Repair Shop Supplier Code,ES - Auto - Supplier Name,ES - Auto - Taxi Supplier Name,ES - Auto - Towing Supplier Name,ES - Auto - Repair Shop Supplier Name,ES - Auto - Towing,ES - Auto - Repair Shop,ES - Auto - Taxi,ES - Auto - Supplier Type Text,ES - Motivo de valoración - Auto Assitance - Corredores,ES - Auto Assitance - Request - Corredores,ES - Detailed Reason - Auto Assitance - Service -Corredores,ES - Detailed Reason - Auto Assitance - Towingtaxi -Corredores,ES - Auto Assitance - Satisfaction - Ease,ES - Auto Assitance - Satisfaction - Info - Corr,ES - Auto Assitance - Satisfaction - Towing - Time - Corr,ES - Auto Assitance - Satisfaction - Taxi - Time - Corr,ES - Auto Assitance - Satisfaction - Towing - Driver - Corr,ES - Auto Assitance - Satisfaction - Kidness - Taxi - Corr,ES - Auto Assitance - Satisfaction - Time -Corr,ES - Motivo de valoración - Auto Claim - Corredores,ES - Claim - Detailed reason - Communication New,ES - Auto -Claim - Detailed reason - Reparacion,ES - Auto -Claim - Detailed reason - Information,ES - Auto -Claim - Detailed reason - Resolution,ES - Auto Claim - Satisfaction Atention,ES - Auto Claim - Satisfaction Next Steps,ES - Auto Claim - Satisfaction Proactivity,ES - Auto Claim - Satisfaction Explanations,ES - Auto Claim - Satisfaction Assesment,ES - Auto Claim - Satisfaction Resolution,ES - Auto Claim - Satisfaction Quality,ES - Auto Claim - Satisfaction Repairer,ES - Home - Compensation Amount,ES - Home - File paid Amount,ES - Home - Number repair Services,ES - Home - Activity Repair Code,ES - Home - Number Control Services,ES - Home - Code Control Activity,ES - Home - Group Tramitation Code,ES - Home - Risk Address Postal Code,ES - Home - Risk Address Town,ES - Home - Risk Address Province,ES - Commerce - Epigraph code,ES - Commerce - Epigraph name,ES - Commerce - Mapfre entity,ES - Commerce - Mapfre Entity Code,ES - Commerce - Santander purchase channel,ES - Commerce - Santander user ID,ES - Commerce - Dossier Zone,ES - Commerce - Expert Name,ES - Commerce - Expert Code,ES - Commerce - Repairer Name,ES - Commerce - Repairer Code,ES - Commerce - PZCIS Code,ES- Motivo de valoración - Hogar - Corredores,ES - Home - Detailed Reason Claim Communication including Corredores,ES - Home - Detailed Reason Claim Evolution including Corredores,ES - Home - Satisfaction Repairers Service including Corredores,ES - Detailed Reason Claim Resolution including Corr,ES - Detailed Reason Compensation Charge including Corredores,ES - Home - Satisfaction Claim Management including Corredores,ES - Home - Satisfaction Information including Corredores,ES - Home - Satisfaction Proactivity including Corredores,ES - Home - Satisfaction Communication including Corredores,ES - Home - Satisfaction 

In [ ]:
df.to_excel(r'Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\NPS_Salud\nps_salud.xlsx', index=False)
print("El archivo 'nps_salud.xlsx' se ha exportado correctamente.")

El archivo 'nps_salud.xlsx' se ha exportado correctamente.


In [3]:
df1 = pd.read_excel(r'Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\NPS_Salud\nps_salud.xlsx')

# Nos quedamos unicamente con las columnas que contienen el id y el comentario
df = df1[
    [
        "Surveyid for internal use (e.g. RI link)",
        "ES - Corredores - Verbatim Comment including Corredores"
    ]
].copy()

# Eliminamos las filas que no tienen comentario
df = df[
    df["ES - Corredores - Verbatim Comment including Corredores"].notna() &
    (df["ES - Corredores - Verbatim Comment including Corredores"].str.strip() != "")
]

# Rseteamos los indices
df = df.reset_index(drop=True)

print("Filas totales:", len(df1))
print("Filas con texto:", len(df))

Filas totales: 19206
Filas con texto: 5575


In [4]:
# Diccionario de aspectos
aspectos_dic = {
    'Atencion_y_Trato': [
        'trato', 'buen trato', 'mal trato',
        'amabilidad', 'amable', 'desagradable', 'antipático',
        'empatía', 'empatia', 'falta de empatía',
        'atención', 'buena atención', 'mala atención',
        'atención deficiente',
        'profesionalidad', 'profesional', 'poco profesional',
        'explicaciones', 'explicación', 'explican bien', 'no explican',
        'información', 'información clara'
    ],
 
    'Gestion_de_Citas': [
        'cita', 'citas', 'pedir cita',
        'agenda', 'agendas cerradas',
        'cita online', 'cita telefónica',
        'modificar cita', 'cancelar cita'
    ],
 
    'Disponibilidad_y_Acceso': [
        'disponibilidad', 'sin disponibilidad',
        'no hay citas',
        'lista de espera', 'espera larga',
        'demora', 'retraso',
        'meses', 'semanas'
    ],
 
    'Cuadro_Medico_y_Cobertura': [
        'cuadro médico', 'pocos médicos', 'falta de especialistas',
        'dermatólogo', 'ginecólogo', 'traumatólogo',
        'no hay especialistas',
        'centros disponibles',
        'cobertura', 'no cubre', 'no lo cubre'
    ],
 
    'Ubicacion_y_Proximidad': [
        'cerca', 'lejano', 'distancia',
        'proximidad', 'cerca de casa',
        'en mi ciudad', 'zona', 'localidad'
    ],
 
    'Tiempos_de_Espera': [
        'espera', 'tiempo de espera',
        'mucho tiempo', 'larga espera',
        'retraso', 'tardanza',
        'urgencias', 'urgente',
        'rápido', 'rapidez',
        'lento'
    ],
 
    'Autorizaciones': [
        'autorización', 'autorizar',
        'pendiente de autorización',
        'denegado', 'rechazo',
        'documentación', 'informes',
        'solicitud', 'trámite'
    ],
 
    'Gestion_Administrativa': [
        'trámites', 'gestión',
        'reembolso', 'factura',
        'copago',
        'póliza', 'poliza',
        'cobertura', 'condiciones'
    ],
 
    'Precio_y_Coste': [
        'precio', 'caro', 'muy caro', 'carísimo',
        'cuota', 'cuota alta',
        'subida', 'subida de precio',
        'prima', 'incremento',
        'coste', 'coste elevado',
        'pagar'
    ],
 
    'Canales_y_Comunicacion': [
        'teléfono', 'llamar', 'no cogen',
        'no responden', 'difícil contactar',
        'email', 'correo',
        'respuesta', 'sin respuesta',
        'comunicación',
        'app', 'aplicación',
        'web', 'página web'
    ]
}

In [5]:
# Normalizamos el texto
import re

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    texto = re.sub(r"\s+", " ", texto)
    return texto


# función de ventana
def extraer_fragmento_local(texto, palabra, ventana=50):
    texto = normalizar_texto(texto)

    idx = texto.find(palabra)
    if idx == -1:
        return None

    inicio = max(0, idx - ventana)
    fin = min(len(texto), idx + len(palabra) + ventana)

    fragmento = texto[inicio:fin]

    # Ajustar bordes a palabra completa
    if inicio > 0:
        primer_espacio = fragmento.find(" ")
        if primer_espacio != -1:
            fragmento = fragmento[primer_espacio + 1:]

    if fin < len(texto):
        ultimo_espacio = fragmento.rfind(" ")
        if ultimo_espacio != -1:
            fragmento = fragmento[:ultimo_espacio]

    fragmento = fragmento.strip(" ,.;:-")

    return fragmento.strip()

def unir_fragmentos(fragmentos):
    """
    Une varios fragmentos en uno solo manteniendo el orden
    """
    if not fragmentos:
        return None

    # eliminar duplicados manteniendo orden
    fragmentos = list(dict.fromkeys(fragmentos))

    # unir como texto continuo
    texto_unido = " ... ".join(fragmentos)

    return texto_unido

# función principal
def obtener_fragmentos_por_aspecto(texto, dic_aspectos):
    resultado = {}

    for aspecto, palabras in dic_aspectos.items():
        fragmentos = []

        for p in palabras:
            frag = extraer_fragmento_local(texto, p)
            if frag:
                fragmentos.append(frag)

        if fragmentos:
            fragmento_unico = unir_fragmentos(fragmentos)
            resultado[aspecto] = fragmento_unico

    return resultado

df["fragmentos_por_aspecto"] = df["ES - Corredores - Verbatim Comment including Corredores"].apply(
    lambda x: obtener_fragmentos_por_aspecto(x, aspectos_dic))

In [6]:
# Cargamos el modelo de analisis de sentimiento
import joblib
model = joblib.load("modelo_sentimiento_logreg.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# Limpiamos el texto
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words("spanish"))

def limpiar_para_modelo(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(texto)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

# Predicción de sentimiento
def predecir_sentimiento(texto):
    texto_limpio = limpiar_para_modelo(texto)
    vec = vectorizer.transform([texto_limpio])
    return model.predict(vec)[0]

# genera TODAS las columnas basadas en el diccionario
def aspectos_a_columnas(dic_fragmentos):
    resultado = {}

    for aspecto in aspectos_dic.keys():  
        fragmento = dic_fragmentos.get(aspecto, None)

        if fragmento:
            resultado[aspecto] = predecir_sentimiento(fragmento)
        else:
            resultado[aspecto] = ""   

    return resultado

# Aplicar
df["aspectos_tmp"] = df["fragmentos_por_aspecto"].apply(aspectos_a_columnas)

# Expandir a columnas
df_aspectos = df["aspectos_tmp"].apply(pd.Series)

# Unir al dataframe original
df = pd.concat([df, df_aspectos], axis=1)

# Eliminar columna temporal
df.drop(columns=["aspectos_tmp"], inplace=True)

In [8]:
# Lista de columnas de aspectos (basada en tu diccionario)
columnas_aspectos = list(aspectos_dic.keys())

df1 = df1.merge(
    df[
        ["Surveyid for internal use (e.g. RI link)"] + columnas_aspectos
    ],
    on="Surveyid for internal use (e.g. RI link)",
    how="left")

In [9]:
df1.to_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\nps_salud_aspecto_sentimiento.xlsx", index=False)